In [0]:
# Install required libraries
# azure-storage-blob: connects to Azure Blob Storage
# openpyxl: required by pandas to read .xlsx Excel files
%pip install azure-storage-blob openpyxl -q
print("Libraries installed successfully!")

In [0]:
# Restart kernel after install to load new libraries
dbutils.library.restartPython()

## Connect & load FocusPOS raw file

In [0]:
# ============================================
# Connect to Azure Blob Storage
# Load FocusPOS raw Excel from bronze
# ============================================
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

# Connect securely via Key Vault — never hardcode credentials
storage_account_key = dbutils.secrets.get(scope="churchs-payroll-kv", key="storage-account-key")
blob_service_client = BlobServiceClient(
    account_url="https://churchspayrollstorage.blob.core.windows.net",
    credential=storage_account_key
)
print("Connected to Azure Blob Storage successfully!")

# Point to FocusPOS file in bronze container
blob_client = blob_service_client.get_blob_client(
    container="bronze",
    blob="focuspos/FocusPOS Input.xlsx"
)

# Download and read Excel without headers
blob_data = blob_client.download_blob().readall()
df_raw_focus = pd.read_excel(io.BytesIO(blob_data), header=None)

# Validation — check file loaded correctly
print("Raw file loaded! Shape:", df_raw_focus.shape)
assert df_raw_focus.shape[0] > 0, "ERROR: Raw file is empty!"
assert df_raw_focus.shape[1] > 0, "ERROR: No columns found!"
print("Validation passed — file has", df_raw_focus.shape[0], "rows and", df_raw_focus.shape[1], "columns")

## Column Index Mapping

In [0]:
# ============================================
# AUTO DETECT: FocusPOS Column Position Discovery
# Purpose: Auto detect column positions
# from raw FocusPOS Excel without manual inspection
# ============================================

import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(message)s")
logger = logging.getLogger(__name__)

def detect_focuspos_columns(df, header_rows=[7, 8, 9], sample_data_row=18, employee_row=17):
    """
    Auto detects column positions from raw FocusPOS Excel.
    Handles merged cells by scanning multiple header rows.
    Validates detected positions against actual data row.
    Returns dictionary of column_name: column_index
    """
    positions = {}

    # Combine all header rows into one label per column
    combined_headers = {}
    for h_row in header_rows:
        for col_idx in range(df.shape[1]):
            val = str(df.iloc[h_row, col_idx]).strip() if pd.notna(df.iloc[h_row, col_idx]) else ""
            val = val.replace("\n", " ").strip()
            if val and val != "nan":
                if col_idx in combined_headers:
                    combined_headers[col_idx] += f" {val}"
                else:
                    combined_headers[col_idx] = val

    # Map known column names to positions from headers
    # Validate each position against actual sample data row
    for col_idx, header in combined_headers.items():
        header_lower = header.lower()
        sample = df.iloc[sample_data_row, col_idx]
        sample_str = str(sample).strip() if pd.notna(sample) else ""

        if "date" in header_lower and "in" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Date_In"] = col_idx
            else:
                for adj in [col_idx + 1, col_idx - 1]:
                    if 0 <= adj < df.shape[1] and pd.notna(df.iloc[sample_data_row, adj]):
                        positions["Date_In"] = adj
                        logger.info(f"Date_In corrected to col[{adj}]")
                        break

        elif "date" in header_lower and "out" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Date_Out"] = col_idx
            else:
                for adj in [col_idx + 1, col_idx - 1]:
                    if 0 <= adj < df.shape[1] and pd.notna(df.iloc[sample_data_row, adj]):
                        positions["Date_Out"] = adj
                        logger.info(f"Date_Out corrected to col[{adj}]")
                        break

        elif "time" in header_lower and "in" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Clock_In"] = col_idx
            else:
                for adj in [col_idx + 1, col_idx - 1]:
                    if 0 <= adj < df.shape[1] and pd.notna(df.iloc[sample_data_row, adj]):
                        positions["Clock_In"] = adj
                        logger.info(f"Clock_In corrected to col[{adj}]")
                        break

        elif "time" in header_lower and "out" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Clock_Out"] = col_idx
            else:
                for adj in [col_idx + 1, col_idx - 1]:
                    if 0 <= adj < df.shape[1] and pd.notna(df.iloc[sample_data_row, adj]):
                        positions["Clock_Out"] = adj
                        logger.info(f"Clock_Out corrected to col[{adj}]")
                        break

        elif "hours" in header_lower and "break" not in header_lower:
            if sample_str and sample_str != "nan":
                positions["Hours"] = col_idx

        elif "rate" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Rate"] = col_idx
            else:
                for adj in [col_idx + 1, col_idx - 1]:
                    if 0 <= adj < df.shape[1] and pd.notna(df.iloc[sample_data_row, adj]):
                        positions["Rate"] = adj
                        logger.info(f"Rate corrected to col[{adj}]")
                        break

        elif "total pay" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Total_Pay"] = col_idx

        elif "tipped" in header_lower and "sales" in header_lower:
            positions["Tipped_Sales"] = col_idx

        elif "charge" in header_lower and "tips" in header_lower:
            positions["Charge_Tips"] = col_idx

        elif "gratuities" in header_lower:
            if sample_str and sample_str != "nan":
                positions["Declared_Tips"] = col_idx
            else:
                for adj in [col_idx + 1, col_idx - 1]:
                    if 0 <= adj < df.shape[1] and pd.notna(df.iloc[sample_data_row, adj]):
                        positions["Declared_Tips"] = adj
                        logger.info(f"Declared_Tips corrected to col[{adj}]")
                        break

        elif "pay" in header_lower and "total" not in header_lower:
            if sample_str and sample_str != "nan":
                positions["Pay"] = col_idx

        elif "breaks" in header_lower:
            positions["Breaks"] = col_idx

    # Detect Store name column — contains Churchs keyword
    for col_idx in range(df.shape[1]):
        for row_idx in range(min(10, df.shape[0])):
            val = str(df.iloc[row_idx, col_idx]).strip() if pd.notna(df.iloc[row_idx, col_idx]) else ""
            if "Churchs" in val or "Church" in val:
                positions["Store"] = col_idx
                positions["Store_Row"] = row_idx
                break

    # Detect ID1 and ID2 by scanning ALL rows
    # ID1: is always at col 5, ID2: is always at col 14
    # Payroll ID value is one column after ID2 label (col 15)
    for row_idx in range(df.shape[0]):
        row_data = df.iloc[row_idx]
        for col_idx in range(df.shape[1]):
            val = str(row_data[col_idx]).strip() if pd.notna(row_data[col_idx]) else ""
            if val == "ID1:":
                positions["ID1_Label"] = col_idx
                logger.info(f"ID1 label found at row_idx={row_idx} col[{col_idx}]")
                for col_idx2 in range(df.shape[1]):
                    val2 = str(row_data[col_idx2]).strip() if pd.notna(row_data[col_idx2]) else ""
                    if val2 == "ID2:":
                        positions["ID2_Label"] = col_idx2
                        positions["Payroll_ID"] = col_idx2 + 1
                        logger.info(f"ID2 label found at col[{col_idx2}] — Payroll ID at col[{col_idx2 + 1}]")
                        break
                break
        if "ID1_Label" in positions:
            break

    # Employee name is always in column 3
    positions["Employee_Name"] = 3

    return positions

# Run auto detection
logger.info("Starting FocusPOS auto column detection...")
col_pos = detect_focuspos_columns(df_raw_focus)

# Display detected positions with correct sample rows
print("\n=== Auto Detected FocusPOS Column Positions ===")
for name, idx in col_pos.items():
    if isinstance(idx, int) and name not in ["Store_Row"]:
        if name in ["ID1_Label", "ID2_Label", "Payroll_ID"]:
            sample = str(df_raw_focus.iloc[17, idx]).strip() if pd.notna(df_raw_focus.iloc[17, idx]) else "NaN"
        elif name == "Store":
            sample = str(df_raw_focus.iloc[2, idx]).strip() if pd.notna(df_raw_focus.iloc[2, idx]) else "NaN"
        else:
            sample = str(df_raw_focus.iloc[18, idx]).strip() if pd.notna(df_raw_focus.iloc[18, idx]) else "NaN"
        print(f"  {name:<20} → col[{idx}] = {sample}")

logger.info("Auto detection complete!")

## Parse hierarchy

In [0]:
# ============================================
# Parse FocusPOS hierarchy
# Structure: Store → Employee → Shift rows
# Uses auto detected column positions from Cell 4
# ============================================

logger.info("Starting FocusPOS hierarchy parsing using auto detected columns...")
logger.info(f"Total raw rows to process: {len(df_raw_focus)}")

records = []
store = employee = payroll = position = None

store_count = 0
employee_count = 0
skipped_rows = 0

for idx, row in df_raw_focus.iterrows():
    c2 = str(row[2]).strip() if pd.notna(row[2]) else ""
    c3 = str(row[col_pos.get("Employee_Name", 3)]).strip() if pd.notna(row[col_pos.get("Employee_Name", 3)]) else ""
    c5 = str(row[col_pos.get("ID1_Label", 5)]).strip() if pd.notna(row[col_pos.get("ID1_Label", 5)]) else ""

    # Store name row — identified by Churchs keyword
    if "Churchs" in c2:
        store = c2
        employee = None
        payroll = None
        position = None
        store_count += 1
        logger.info(f"Store found: {store}")

    # Employee name and payroll ID row
    # ID1 = SSN (ignored for PII), ID2 = payroll ID
    elif c3 and "ID1:" in c5:
        employee = c3
        payroll = str(row[col_pos.get("Payroll_ID", 15)]).strip() if pd.notna(row[col_pos.get("Payroll_ID", 15)]) else ""
        employee_count += 1
        logger.info(f"Employee found: {employee} | Payroll ID: {payroll} | Store: {store}")

    # Shift data row — has valid datetime in Date_In column
    elif pd.notna(row[col_pos.get("Date_In", 5)]) and "2026" in str(row[col_pos.get("Date_In", 5)]):
        try:
            records.append({
                "StoreID":        store,
                "Employee_name":  employee,
                "Payroll_ID":     payroll,
                "Position":       c3,
                "Shift_Date":     pd.to_datetime(row[col_pos.get("Date_In",      5)], errors="coerce"),
                "Date_Out":       pd.to_datetime(row[col_pos.get("Date_Out",      8)], errors="coerce"),
                "Clock_In":       row[col_pos.get("Clock_In",      11)],
                "Clock_Out":      row[col_pos.get("Clock_Out",     14)],
                "Hours":          pd.to_numeric(row[col_pos.get("Hours",          17)], errors="coerce"),
                "Rate":           pd.to_numeric(row[col_pos.get("Rate",           19)], errors="coerce"),
                "Pay":            pd.to_numeric(row[col_pos.get("Pay",            21)], errors="coerce"),
                "Total_Pay":      pd.to_numeric(row[col_pos.get("Total_Pay",      24)], errors="coerce"),
                "Tipped_Sales":   pd.to_numeric(row[col_pos.get("Tipped_Sales",   25)], errors="coerce"),
                "Charge_Tips":    pd.to_numeric(row[col_pos.get("Charge_Tips",    28)], errors="coerce"),
                "Declared_Tips":  pd.to_numeric(row[col_pos.get("Declared_Tips",  33)], errors="coerce")
            })
        except Exception as e:
            skipped_rows += 1
            logger.error(f"ERROR: Failed to parse row {idx} — {e}")
            logger.error(f"Row data: {row.to_dict()}")

logger.info("Parsing loop complete!")

# Build silver DataFrame
df_silver_focus = pd.DataFrame(records)
logger.info(f"DataFrame created with {len(df_silver_focus)} records and {len(df_silver_focus.columns)} columns")

# Fix week_end_date — Saturday of each shift week
df_silver_focus["week_end_date"] = df_silver_focus["Shift_Date"].apply(
    lambda d: (d + pd.Timedelta(days=(5 - d.weekday()) % 7)).date() if pd.notna(d) else None
)
logger.info("week_end_date calculated successfully")

# Fix Shift_Date to date only
df_silver_focus["Shift_Date"] = df_silver_focus["Shift_Date"].dt.date
logger.info("Shift_Date converted to date only")

# Logs
print("\n=== Parse Summary ===")
print("Stores found:", store_count)
print("Employees found:", employee_count)
print("Shift records parsed:", len(records))
print("Rows skipped:", skipped_rows)

# Validation checks
print("\n=== Validation Checks ===")
try:
    assert len(df_silver_focus) > 0, "ERROR: No records parsed!"
    assert df_silver_focus["StoreID"].notna().all(), "ERROR: Missing StoreID values!"
    assert df_silver_focus["Employee_name"].notna().all(), "ERROR: Missing Employee names!"
    assert df_silver_focus["Payroll_ID"].notna().all(), "ERROR: Missing Payroll IDs!"
    assert df_silver_focus["Hours"].notna().sum() > 0, "ERROR: No valid Hours!"
    logger.info("All validation checks passed!")
    print("All validation checks passed!")
except AssertionError as e:
    logger.error(f"VALIDATION FAILED: {e}")
    raise

print("\nUnique stores:", df_silver_focus["StoreID"].nunique())
print("Unique employees:", df_silver_focus["Payroll_ID"].nunique())
print("Date range:", df_silver_focus["Shift_Date"].min(), "to", df_silver_focus["Shift_Date"].max())
print("Hours range:", df_silver_focus["Hours"].min(), "to", df_silver_focus["Hours"].max())
print("\nSample:")
print(df_silver_focus.head(5).to_string())

## Upload Cleaned FocusPOS Data to Silver Container

In [0]:
# ============================================
# Upload cleaned FocusPOS data
# to silver container as CSV
# ============================================

logger.info("Starting upload of FocusPOS silver CSV to blob storage...")

# Convert DataFrame to CSV in memory
csv_buffer = io.BytesIO()
df_silver_focus.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)
logger.info(f"CSV buffer created — {len(df_silver_focus)} records ready for upload")

# Upload to silver container
silver_client = blob_service_client.get_blob_client(
    container="silver",
    blob="focuspos/FocusPOS_Silver.csv"
)

try:
    silver_client.upload_blob(csv_buffer, overwrite=True)
    logger.info("FocusPOS Silver CSV uploaded successfully!")
    logger.info("Location: silver/focuspos/FocusPOS_Silver.csv")
except Exception as e:
    logger.error(f"ERROR: Failed to upload FocusPOS Silver CSV — {e}")
    raise

# Validation — verify upload
try:
    assert len(df_silver_focus) > 0, "ERROR: Nothing was written to silver!"
    logger.info("Upload validation passed!")
    print("\nFocusPOS Silver CSV uploaded successfully!")
    print("Location: silver/focuspos/FocusPOS_Silver.csv")
    print("Total records written:", len(df_silver_focus))
    print("Columns written:", df_silver_focus.columns.tolist())
    print("Upload validation passed!")
except AssertionError as e:
    logger.error(f"UPLOAD VALIDATION FAILED: {e}")
    raise

In [0]:
df_silver_focus.shape
df_silver_focus.head
df_silver_focus.head(166)
